# 04 — Evaluate on ARC-AGI with Test-Time Training (TTT)

**Goal:** measure performance on the official ARC-AGI evaluation set, with the single trick that has consistently been the biggest win in the ARC Prize: **test-time training**.

**TTT in 1 paragraph:** for each test task, take the 2–4 train examples for that *single* task, apply augmentations to multiply them ~30x, and fine-tune a small LoRA adapter for ~10–30 steps on top of the base model. Then predict the test output. Cost: ~30s per task on a T4. Win: typically +15–25 percentage points.

You can run this notebook on a single T4 / RTX-class GPU, locally or on Kaggle. Use the distilled 3B student from notebook 03 for speed; or the 14B teacher from notebook 02 if you have an H100 around.


In [ ]:
# Hardware
import torch, os
print("CUDA:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
!pip -q install -U transformers==4.46.2 accelerate==1.1.1 bitsandbytes==0.44.1 \
    datasets==3.1.0 peft==0.13.2 sentencepiece


In [ ]:
# 1) ARC eval set
import json, urllib.request, zipfile, io
from pathlib import Path
DATA_DIR = Path("./arc_data"); DATA_DIR.mkdir(exist_ok=True)
if not (DATA_DIR / "evaluation").exists():
    url = "https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip"
    with urllib.request.urlopen(url) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(DATA_DIR)
    src = DATA_DIR / "ARC-AGI-master" / "data"
    (DATA_DIR / "evaluation").mkdir(exist_ok=True)
    for f in (src / "evaluation").glob("*.json"):
        (DATA_DIR / "evaluation" / f.name).write_bytes(f.read_bytes())
arc_eval = [(p.stem, json.loads(p.read_text())) for p in sorted((DATA_DIR/"evaluation").glob("*.json"))]
print("Eval tasks:", len(arc_eval))


In [ ]:
# 2) Load base model + your trained adapter
# Pick ONE: the distilled student (cheaper) or the 14B teacher (stronger).
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_ID = "Qwen/Qwen2.5-3B-Instruct"           # student
# BASE_ID = "Qwen/Qwen2.5-14B-Instruct"        # teacher (needs >=24GB VRAM in 4-bit)
ADAPTER_PATH = "./student_out/adapter"          # path to the LoRA you trained

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)

tok = AutoTokenizer.from_pretrained(BASE_ID)
if tok.pad_token is None: tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb,
                                            device_map="auto", torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, ADAPTER_PATH, is_trainable=False)
model.eval()
print("Loaded model + adapter.")


In [ ]:
# 3) Shared helpers (mirror notebook 02)
import copy, random
SYSTEM = ("You are an expert ARC puzzle solver. Output only the test output grid as digits "
          "separated by spaces, one row per line. No explanations.")

def grid_to_str(g): return "\n".join(" ".join(str(c) for c in row) for row in g)

def parse_grid(s):
    rows = []
    for line in s.strip().splitlines():
        line = line.strip()
        if not line: continue
        try: rows.append([int(c) for c in line.split()])
        except: return None
    if not rows or any(len(r)!=len(rows[0]) for r in rows): return None
    return rows

def format_user(task):
    parts = ["You are solving an ARC puzzle. Each grid is a 2D array of digits 0-9 (colors).",
             "Find the transformation rule from the input/output examples, then apply it to the test input.", ""]
    for i, ex in enumerate(task["train"], 1):
        parts.append(f"Example {i} Input:");  parts.append(grid_to_str(ex["input"]))
        parts.append(f"Example {i} Output:"); parts.append(grid_to_str(ex["output"]))
        parts.append("")
    parts.append("Test Input:"); parts.append(grid_to_str(task["test"][0]["input"]))
    parts.append(""); parts.append("Test Output:")
    return "\n".join(parts)

def format_assistant(task):
    return grid_to_str(task["test"][0]["output"])

# D4 + color permutations for TTT augmentation
def rot90(g):  return [list(r) for r in zip(*g[::-1])]
def fliplr(g): return [r[::-1] for r in g]
D4 = [lambda g: g, rot90,
      lambda g: rot90(rot90(g)),
      lambda g: rot90(rot90(rot90(g))),
      fliplr,
      lambda g: rot90(fliplr(g)),
      lambda g: rot90(rot90(fliplr(g))),
      lambda g: rot90(rot90(rot90(fliplr(g))))]

def aug_task(task, k, color_perm):
    f = D4[k]
    def apply(ex):
        return {"input":  [[color_perm[c] for c in row] for row in f(ex["input"])],
                "output": [[color_perm[c] for c in row] for row in f(ex["output"])]}
    return {"train":[apply(e) for e in task["train"]],
            "test": [apply(e) for e in task["test"]]}


In [ ]:
# 4) Inference helper (greedy)
@torch.inference_mode()
def predict(m, task, max_new=2048):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":format_user(task)}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(m.device)
    out = m.generate(**inp, max_new_tokens=max_new, do_sample=False,
                     pad_token_id=tok.eos_token_id)
    return parse_grid(tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True))


In [ ]:
# 5) TTT — for each task, train a fresh LoRA on its own examples.
# We make a *separate* small adapter on top of the loaded model, train ~20 steps,
# predict, then discard the adapter.

from peft import LoraConfig, get_peft_model
from torch.optim import AdamW

def build_ttt_examples(task, n_aug=20, rng=None):
    # Build (prompt, completion) chat strings from augmented versions of THIS task,
    # treating each train example as a held-out test to learn from.
    rng = rng or random.Random(0)
    out = []
    examples = task["train"]
    if len(examples) < 2:
        return out
    for _ in range(n_aug):
        k = rng.randrange(8)
        perm = list(range(10)); rng.shuffle(perm)
        a = aug_task(task, k, perm)
        # Hold out one example to be the 'test', use others as train
        for held in range(len(a["train"])):
            sub = {"train":[e for j,e in enumerate(a["train"]) if j != held],
                   "test":[a["train"][held]]}
            msgs = [{"role":"system","content":SYSTEM},
                    {"role":"user","content":format_user(sub)},
                    {"role":"assistant","content":format_assistant(sub)}]
            out.append(tok.apply_chat_template(msgs, tokenize=False))
    return out

def ttt_and_predict(task, steps=20, lr=5e-5, n_aug=20):
    # Run TTT for one task, return prediction grid (or None on failure).
    # Wrap the already-PEFT model with a SECOND LoRA via peft is messy; instead,
    # train the existing adapter further but on a *copy* of its state, then restore.
    # Snapshot adapter weights:
    snap = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
    # Re-enable training on adapter params
    for n, p in model.named_parameters():
        p.requires_grad = (n in snap)
    model.train()

    texts = build_ttt_examples(task, n_aug=n_aug)
    if not texts:
        model.eval()
        return predict(model, task)

    opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    rng = random.Random(0)
    for step in range(steps):
        t = rng.choice(texts)
        enc = tok(t, return_tensors="pt", truncation=True, max_length=3500).to(model.device)
        labels = enc.input_ids.clone()
        out = model(**enc, labels=labels)
        out.loss.backward()
        opt.step(); opt.zero_grad()

    model.eval()
    pred = predict(model, task)

    # Restore adapter weights so the next task starts from the same base
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in snap:
                p.copy_(snap[n])
    return pred


In [ ]:
# 6) Run eval — start without TTT for a sanity number, then with TTT.
import time

# (a) No TTT
N = 50  # ramp to len(arc_eval) for full run
t0 = time.time()
correct_no_ttt = 0
for i, (name, task) in enumerate(arc_eval[:N]):
    pred = predict(model, task)
    if pred == task["test"][0]["output"]:
        correct_no_ttt += 1
print(f"No-TTT: {correct_no_ttt}/{N} = {correct_no_ttt/N:.1%}  ({time.time()-t0:.0f}s)")


In [ ]:
# (b) With TTT — ~30s/task on T4. Budget accordingly.
import time
t0 = time.time()
correct_ttt = 0
for i, (name, task) in enumerate(arc_eval[:N]):
    pred = ttt_and_predict(task, steps=20, lr=5e-5, n_aug=20)
    if pred == task["test"][0]["output"]:
        correct_ttt += 1
    if (i+1) % 5 == 0:
        elapsed = time.time() - t0
        print(f"[{i+1}/{N}] TTT acc {correct_ttt}/{i+1} = {correct_ttt/(i+1):.1%}  ({elapsed:.0f}s, ~{elapsed/(i+1):.1f}s/task)")
print(f"\nWith TTT: {correct_ttt}/{N} = {correct_ttt/N:.1%}")


In [ ]:
# 7) Save predictions in ARC submission format
import json
preds = {}
for name, task in arc_eval[:N]:
    pred = ttt_and_predict(task, steps=20, lr=5e-5, n_aug=20)
    if pred is None:
        pred = [[0]]
    preds[name] = [{"attempt_1": pred, "attempt_2": pred}]
with open("submission.json","w") as f:
    json.dump(preds, f)
print("Wrote submission.json with", len(preds), "tasks.")
